[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/eigen_fluids/eigen_fluids.ipynb)

# Laplacian Eigenfunction Fluids

Laplacian eigenfunction fluids evolve a thin incompressible fluid film over a curved triangulated surface through the amplitudes of smooth vorticity patterns, the Laplace–Beltrami eigenmodes of the surface. A streamfunction acts as a height field on the surface whose level curves guide the fluid at a speed proportional to the local slope. Vortices carry one another along through a fixed table of mode interactions determined once from the surface geometry, so each time step updates only the modal amplitudes while viscosity damps finer modes faster than broad currents.

On each triangle, a scalar field's simplex gradient is an extrinsic plane, and `gradient` maps vertex scalars to these face planes. Meeting the streamfunction's gradient plane with the unit tangent patch gives the intrinsic tangent velocity `gradient(stream) ^ normals` along the level curve. Meeting a mode's gradient plane with the unnormalized area patch gives its area-weighted flow, and pairing that flow against another field's gradient plane via `&` integrates their surface Jacobian over the triangle. The metric enters only in the corner simplex gradients, the unit tangent patches, and the Dirichlet stiffness `gradient.adjugate()(medium(gradient))`, whose generalized eigenvectors against the lumped vertex mass are the modes. The modal formulation follows De Witt, Lessig and Fiume, [*Fluid Simulation using Laplacian Eigenfunctions*](https://www.dgp.toronto.edu/~tyler/eigenfluids/) (2012), and Liu, Mason, Tong and Desbrun, [*Model-Reduced Variational Fluid Simulation*](https://geometry.caltech.edu/pubs/LMTD15.pdf) (2015).

In vector-calculus and finite-element matrix notation on a curved surface $\Gamma$ with unit normal $\mathbf n$, the film obeys $\partial_t\omega + (\mathbf n \times \nabla_\Gamma\psi)\cdot\nabla_\Gamma\omega = \nu\Delta_\Gamma\omega$ with $\omega = -\Delta_\Gamma\psi$; expanding in generalized eigenmodes $K\phi_k = \lambda_k M\phi_k$ yields $\dot\omega_k = \sum_{i,j} C_{kij}(\omega_i/\lambda_i)\omega_j - \nu\lambda_k\omega_k$ with structure constants $C_{kij} = \int_\Gamma \phi_k\,(\mathbf n \times \nabla_\Gamma\phi_i)\cdot\nabla_\Gamma\phi_j\,dA$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import save_animation, save_figure, show_animation
from numga.topology import Topology
from examples.fluids.eigen_fluids import render
from examples.fluids.eigen_fluids.scenarios import bunny

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Streamfunction simplex gradients on triangles are extrinsic planes orthogonal to the gradient direction.
Plane = ga.gatype.vector()                          # Vector+
# Vertex positions, edge displacements, and tangent fluid velocities are intrinsic antivectors.
Point = ga.gatype.antivector()                      # Antivector-
# Oriented triangle area elements and unit tangent patches are intrinsic joins of two edges.
Patch = ga.gatype.antibivector()                    # Antibivector-
# Vertex-integrated surface densities carry the intrinsic orientation of the pseudoscalar dual.
Antiscalar = ga.gatype.antiscalar()                 # Antiscalar-
GradientMap = ga.gatype((Plane, Scalar))
Mass = ga.gatype((Antiscalar, Scalar))
Superposition = ga.gatype((Scalar, Scalar))
Bracket = ga.gatype((Scalar, Scalar, Scalar))

## Flow modes on the bunny

A streamfunction acts as a height field whose contours guide the flow, with steeper slopes driving faster motion along each contour. Its smoothest spatial patterns are the Laplace–Beltrami eigenmodes, and the triangulated surface determines where broad currents and finer eddies fit.

In [ ]:
topology, vertices = bunny()
save_figure(render.draw_surface(topology, vertices), "eigen_fluids_surface");

In [ ]:
# Count of non-constant Laplace–Beltrami modes and representative mode indices shown below.
n_modes = 128
n_shown_modes = 4
shown_modes = np.array([0, 7, 31, 119])

In [ ]:
# Opposite triangle sides and oriented face patches determine the corner simplex gradients.
edges = topology.d[0] * vertices                                                # Point-[E]
sides = topology.boundary[2, 1] * edges                                         # [3] Point-[F]
patches = (sides[0] & sides[1]) / 2                                             # Patch-[F]
areas = patches.norm()                                                          # Scalar+[F]
normals = patches / areas                                                       # Patch-[F]

# Corner simplex gradients assemble into the map from vertex scalars to face gradient planes.
corner_gradients = (sides | patches.inverse()) / 2                              # [3] Plane+[F]
gradient = (spdiag(corner_gradients * Scalar) * topology.incidence[2, 0]).sum(axis=0)   # (Plane+ <- Scalar+)[F <- V]
# Face-to-corner averaging and lumped dual vertex areas (one third of each incident triangle).
corner_mean = topology.incidence[2, 0].mean(axis=0)                             # Scalar+[F <- V]
dual_areas = corner_mean.adjugate() * areas                                               # Scalar+[V]

# Dirichlet stiffness and lumped vertex mass define the generalized Laplace–Beltrami eigenproblem.
medium = areas * Plane.dual()                                                   # (Point- <- Plane+)[F]
stiffness = gradient.adjugate()(medium(gradient))                               # (Antiscalar- <- Scalar+)[V <- V]
mass = spdiag(dual_areas * Scalar.dual())                                       # (Antiscalar- <- Scalar+)[V <- V]

The modes are the eigenvectors of the stiffness against the mass, the smoothest first; the constant field costs no energy and carries no flow.

In [ ]:
# Solve for the lowest Laplace–Beltrami eigenmodes and drop the constant null mode.
eigenvalues, eigenfields = stiffness.eigh(mass, n_modes + 1)                    # [n_modes + 1] Scalar+, [n_modes + 1] Scalar+[V]
eigenvalues, modes = eigenvalues[1:].field(), eigenfields[1:]                   # Scalar+[n_modes], [n_modes] Scalar+[V]
# Modal superposition maps modal amplitudes onto vertex scalar fields.
superposition = (modes * Scalar).batch().field(0, 1).adjoint()                  # (Scalar+ <- Scalar+)[V <- n_modes]

# Meeting each mode's gradient plane with the unit tangent patch gives its divergence-free face velocity.
mode_velocities = gradient(modes[shown_modes]) ^ normals                        # [n_shown_modes] Point-[F]
save_figure(render.draw_modes(topology, vertices, modes[shown_modes], mode_velocities), "eigen_fluids_modes");

Warm and cool colours mark opposite signs of each mode's streamfunction, and the dark contours trace its streamlines. The modes progress from a broad global dipole across the body to finer standing eddies shaped by the ears, neck, and limbs.

In [ ]:
print("selected mode eigenvalues:", np.round(eigenvalues.batch()[shown_modes].to_array(), 3))

## Mode interactions

Nonlinear advection of vorticity by the velocity field forms the Lie–Poisson bracket of streamfunction and vorticity. Pairing a field's simplex gradient plane with the area-weighted tangent flow via `&` integrates its directional rate of change over each triangle, so in the modal basis the bracket reduces to one bilinear map `structure(stream, vorticity)` computed once from the surface geometry.

In [ ]:
def structure_constants(topology: Topology, patches: Patch, gradient: GradientMap, modes: Scalar) -> Bracket:
    """Each modal Lie–Poisson structure constant integrates a mode against the surface Jacobian of two others, so
    applying the tensor to a modal streamfunction and modal vorticity gives every mode's advection rate."""
    # Averaging operator from triangle corners onto faces.
    corner_mean = topology.incidence[2, 0].mean(axis=0)                         # Scalar+[F <- V]
    face_gradients = gradient(modes)                                            # [n_modes] Plane+[F]
    # Meeting each mode's gradient with the triangle's area patch gives its area-weighted tangent flow,
    # which pairs with another gradient to integrate their Jacobian over the triangle.
    flows = face_gradients ^ patches                                            # [n_modes] Point-[F]
    jacobians = face_gradients[:, None] & flows[None, :]                        # [n_modes, n_modes] Scalar+[F]
    # Because the Jacobian is constant on each triangle, each mode integrates against it via its corner mean.
    face_means = corner_mean * modes                                            # [n_modes] Scalar+[F]
    integral = (face_means * Scalar).batch().field(0, 1)                        # (Scalar+ <- Scalar+)[n_modes <- F]
    # Total antisymmetry preserves cyclic permutations of the three mode indices, placing the vorticity mode last.
    return (integral(jacobians).batch() * Scalar * Scalar).field(0, 1, 2)       # (Scalar+ <- (Scalar+, Scalar+))[n_modes <- (n_modes, n_modes)]


structure = structure_constants(topology, patches, gradient, modes)             # (Scalar+ <- (Scalar+, Scalar+))[n_modes <- (n_modes, n_modes)]

## Interacting vortices

Nearby patches of opposite circulation form vortex dipoles; their interaction drives motion across the curved surface.

The animation cell takes about six seconds, most of it drawing the frames.

In [ ]:
# Opposing vortex dipoles placed across the bunny's shoulder, chest, flank, haunch, neck, and back.
n_vortices = 12
vortex_centres = mv.vector([
    [0.35, -0.54, -0.03],
    [0.40, -0.69, -0.29],
    [-0.08, -0.52, 0.03],
    [-0.16, -0.58, -0.27],
    [-0.54, -0.48, 0.02],
    [-0.45, -0.58, -0.39],
    [0.34, -0.31, 0.26],
    [0.09, -0.40, 0.20],
    [0.22, 0.34, -0.16],
    [-0.14, 0.28, -0.08],
    [-0.48, 0.26, -0.14],
    [0.06, 0.47, -0.30],
]).dual()                                           # [n_vortices] Point-
vortex_strengths = np.array([75, -75, -75, 75, 70, -70, -65, 65, 72, -72, 68, -68])
vortex_radius = 0.12
eigenvalue_cutoff = 140.0

# Fourth-order Runge–Kutta step size, total step count, frame stride, kinematic viscosity, and frame duration.
dt = 0.007
n_steps = 800
stride = 8
playback_rate = 0.5
playback_stride = int(stride * playback_rate)
n_flow_steps = int(n_steps * playback_rate)
viscosity = 0.0009
frame_duration_ms = 40

In [ ]:
def vorticity_rate(vorticity: Scalar, structure: Bracket, eigenvalues: Scalar, viscosity: float) -> Scalar:
    """The modal vorticity amplitudes change at the rate given by the Lie–Poisson advection bracket minus viscous
    decay."""
    stream = vorticity / eigenvalues                                            # Scalar+[n_modes]
    advection = structure(stream, vorticity)                                    # Scalar+[n_modes]
    return advection - viscosity * eigenvalues * vorticity                      # Scalar+[n_modes]


def advect(initial: Scalar, structure: Bracket, eigenvalues: Scalar, viscosity: float,
           dt: float, n_steps: int) -> Iterator[Scalar]:
    """Successive modal vorticity amplitudes advance through classical fourth-order Runge–Kutta steps."""
    vorticity = initial                                                         # Scalar+[n_modes]
    yield vorticity
    for _ in range(n_steps):
        first = vorticity_rate(vorticity, structure, eigenvalues, viscosity)    # Scalar+[n_modes]
        second = vorticity_rate(vorticity + (dt / 2) * first, structure, eigenvalues, viscosity)   # Scalar+[n_modes]
        third = vorticity_rate(vorticity + (dt / 2) * second, structure, eigenvalues, viscosity)   # Scalar+[n_modes]
        fourth = vorticity_rate(vorticity + dt * third, structure, eigenvalues, viscosity)   # Scalar+[n_modes]
        vorticity = vorticity + (dt / 6) * (first + 2 * second + 2 * third + fourth)   # Scalar+[n_modes]
        yield vorticity


def initial_vorticity(vertices: Point, modes: Scalar, eigenvalues: Scalar, mass: Mass) -> Scalar:
    """Localized Gaussian vortex dipoles project onto modal vorticity amplitudes with a smooth high-wavenumber taper."""
    squared_distances = (vertices - vortex_centres).norm_squared()              # [n_vortices] Scalar+[V]
    spots = ((squared_distances * (-0.5 / vortex_radius ** 2)).exp() * vortex_strengths).sum(axis=0)   # Scalar+[V]
    taper = (-0.5 * (eigenvalues / eigenvalue_cutoff) ** 2).exp()               # Scalar+[n_modes]
    return (modes & mass(spots)).sites.sum().field() * taper                    # Scalar+[n_modes]


start = initial_vorticity(vertices, modes, eigenvalues, mass)                   # Scalar+[n_modes]
inviscid = stack(tuple(advect(start, structure, eigenvalues, 0.0, dt, n_steps)))   # [n_steps + 1] Scalar+[n_modes]

In [ ]:
def reconstruct(states: Iterable[Scalar], superposition: Superposition, eigenvalues: Scalar,
                gradient: GradientMap, normals: Patch) -> Iterator[tuple[Scalar, Scalar, Point]]:
    """Each modal vorticity state determines the vertex vorticity, vertex streamfunction, and face tangent velocity."""
    for amplitudes in states:
        vorticity = superposition(amplitudes)                                   # Scalar+[V]
        stream = superposition(amplitudes / eigenvalues)                        # Scalar+[V]
        velocity = gradient(stream) ^ normals                                   # Point-[F]
        yield vorticity, stream, velocity


frames = reconstruct(inviscid[:n_flow_steps + 1:playback_stride], superposition, eigenvalues, gradient, normals)
show_animation(render.animate_flow(topology, vertices, frames), "eigen_fluids_bunny", frame_duration_ms)

Warm and cool patches mark opposite circulation. Dark seeds fade into plumes downstream along the velocity. Streak length is proportional to speed on one fixed scale; still regions keep only specks. The evolving amplitudes describe the whole surface, so a local vortex can interact with motion far from its starting point.

## Energy transfer and viscous decay

The mode interactions redistribute energy and enstrophy, the integrated squared vorticity. Their total antisymmetry makes the inviscid rate pair to zero with both streamfunction and vorticity. Viscosity damps modes in proportion to their eigenvalues, removing fine structure more quickly than broad currents.

In [ ]:
# Compare the inviscid trajectory against a viscous trajectory from the same initial state.
viscous = stack(tuple(advect(start, structure, eigenvalues, viscosity, dt, n_steps)))   # [n_steps + 1] Scalar+[n_modes]
viscous_frames = reconstruct(viscous[:n_flow_steps + 1:playback_stride], superposition, eigenvalues, gradient, normals)
save_animation(render.animate_flow(topology, vertices, viscous_frames), "eigen_fluids_bunny_viscous", frame_duration_ms)

runs = stack((inviscid, viscous))                                               # [2, n_steps + 1] Scalar+[n_modes]
energies = (runs.squared() / eigenvalues).sites.sum() / 2                       # [2, n_steps + 1] Scalar+
enstrophies = runs.squared().sites.sum() / 2                                    # [2, n_steps + 1] Scalar+
inviscid_energy, viscous_energy = energies                                      # [n_steps + 1] Scalar+, [n_steps + 1] Scalar+
inviscid_enstrophy, viscous_enstrophy = enstrophies                             # [n_steps + 1] Scalar+, [n_steps + 1] Scalar+

times = np.arange(n_steps + 1) * dt
save_figure(render.draw_conservation(times, inviscid_energy, viscous_energy, inviscid_enstrophy,
                                     viscous_enstrophy, start, inviscid[-1], viscous[-1]),
            "eigen_fluids_conservation");

In the left panel, the solid and dashed dark curves track inviscid kinetic energy and enstrophy, while the green and orange curves show viscous kinetic energy and viscous enstrophy. In the right panel, the dashed grey curve shows the initial cumulative enstrophy across the modes alongside the final inviscid curve in dark blue and the final viscous curve in orange. Because enstrophy weights high-eigenvalue modes more strongly than kinetic energy does, viscosity drains a much larger share of the enstrophy while cascading vortex interactions push inviscid enstrophy toward finer modes.

## Shear layers on a sphere

Opposing zonal bands of vorticity make shear-layer roll-up easy to follow on a sphere. A smooth meridional ripple disturbs both bands, and the evolving streaks trace how the shear folds each band into a chain of vortices. Even on a finely triangulated sphere, the reduced RK4 integrator steps only the modal amplitudes through the precomputed structure constants and eigenvalues.

The animation cell at the end of this section takes about seven seconds, most of it drawing the frames.

In [ ]:
# Spherical harmonic bands through degree eight on a four-fold subdivided icosphere.
n_sphere_levels = 4
n_sphere_modes = 80
sphere_latitude = 0.44
sphere_width = 0.16
sphere_strength = 9.0
sphere_wave_amplitude = 0.065
sphere_dt = 0.025
n_sphere_steps = 400
sphere_stride = 4
sphere_viscosity = 0.001

In [ ]:
sphere_topology, sphere_coordinates = shapes.sphere(n_sphere_levels)
sphere_vertices = mv.vector(sphere_coordinates).dual().field()                  # Point-[V]
save_figure(render.draw_surface(sphere_topology, sphere_vertices), "eigen_fluids_sphere_surface");

In [ ]:
# Opposite triangle sides and oriented face patches determine the corner simplex gradients.
sphere_edges = sphere_topology.d[0] * sphere_vertices                           # Point-[E]
sphere_sides = sphere_topology.boundary[2, 1] * sphere_edges                    # [3] Point-[F]
sphere_patches = (sphere_sides[0] & sphere_sides[1]) / 2                        # Patch-[F]
sphere_areas = sphere_patches.norm()                                            # Scalar+[F]
sphere_normals = sphere_patches / sphere_areas                                  # Patch-[F]
sphere_corner_gradients = (sphere_sides | sphere_patches.inverse()) / 2         # [3] Plane+[F]
sphere_gradient = (spdiag(sphere_corner_gradients * Scalar) * sphere_topology.incidence[2, 0]).sum(axis=0)   # (Plane+ <- Scalar+)[F <- V]
# Face-to-corner averaging and lumped dual vertex areas (one third of each incident triangle).
sphere_dual_areas = sphere_topology.incidence[2, 0].mean(axis=0).adjugate() * sphere_areas   # Scalar+[V]

# Dirichlet stiffness and lumped vertex mass define the generalized Laplace–Beltrami eigenproblem.
sphere_medium = sphere_areas * Plane.dual()                                     # (Point- <- Plane+)[F]
sphere_stiffness = sphere_gradient.adjugate()(sphere_medium(sphere_gradient))   # (Antiscalar- <- Scalar+)[V <- V]
sphere_mass = spdiag(sphere_dual_areas * Scalar.dual())                         # (Antiscalar- <- Scalar+)[V <- V]
sphere_eigenvalues, sphere_eigenfields = sphere_stiffness.eigh(sphere_mass, n_sphere_modes + 1)   # [n_sphere_modes + 1] Scalar+, [n_sphere_modes + 1] Scalar+[V]
sphere_eigenvalues, sphere_modes = sphere_eigenvalues[1:].field(), sphere_eigenfields[1:]   # Scalar+[n_sphere_modes], [n_sphere_modes] Scalar+[V]
sphere_superposition = (sphere_modes * Scalar).batch().field(0, 1).adjoint()    # (Scalar+ <- Scalar+)[V <- n_sphere_modes]
sphere_structure = structure_constants(sphere_topology, sphere_patches, sphere_gradient, sphere_modes)   # (Scalar+ <- (Scalar+, Scalar+))[n_sphere_modes <- (n_sphere_modes, n_sphere_modes)]

In [ ]:
def shear_layers(vertices: Point, dual_areas: Scalar, latitude: float, width: float,
                 strength: float, wave_amplitude: float) -> Scalar:
    """Opposing zonal vorticity bands on the unit sphere carry a smooth meridional ripple at zero net circulation."""
    heights = mv.z & vertices                                                   # Scalar+[V]
    n_meridians = 3
    angles = np.pi * np.arange(n_meridians) / n_meridians
    meridians = (mv.xy * (-angles / 2)).exp() >> mv.x                           # [n_meridians] Plane+
    first, second, third = meridians & vertices                                 # Scalar+[V], Scalar+[V], Scalar+[V]
    # Multiplying three meridian plane readings makes the azimuthal ripple vanish smoothly at both poles.
    shift = -4 * wave_amplitude * first * second * third                        # Scalar+[V]
    north = (heights - np.sin(latitude) - shift) / width                        # Scalar+[V]
    south = (heights + np.sin(latitude) - shift) / width                        # Scalar+[V]
    vorticity = strength * ((-0.5 * north.squared()).exp() - (-0.5 * south.squared()).exp())   # Scalar+[V]
    return vorticity - (dual_areas * vorticity).sites.sum() / dual_areas.sites.sum()


shear_field = shear_layers(sphere_vertices, sphere_dual_areas, sphere_latitude, sphere_width,
                           sphere_strength, sphere_wave_amplitude)              # Scalar+[V]
sphere_start = (sphere_modes & sphere_mass(shear_field)).sites.sum().field()    # Scalar+[n_sphere_modes]
sphere_states = islice(advect(sphere_start, sphere_structure, sphere_eigenvalues, sphere_viscosity, sphere_dt,
                              n_sphere_steps), 0, None, sphere_stride)
sphere_frames = reconstruct(sphere_states, sphere_superposition, sphere_eigenvalues, sphere_gradient, sphere_normals)
show_animation(render.animate_flow(sphere_topology, sphere_vertices, sphere_frames), "eigen_fluids_sphere",
               frame_duration_ms)

Warm and cool bands mark opposite circulation. The downstream streaks, initially parallel to the bands, turn into eddies as the flow folds the vorticity. Their length shows speed on a fixed scale; their density and opacity stay fixed.

In [ ]:
# checks
# Laplace–Beltrami eigenmodes are orthonormal under the lumped vertex mass form.
gram = (modes[:, None] & mass(modes[None, :])).sites.sum()                      # [n_modes, n_modes] Scalar+
assert np.all((gram - np.eye(n_modes)).abs() < 1e-12)
# Every mode velocity and reconstructed flow velocity lies in its triangle's tangent plane.
assert np.all((mode_velocities & patches).norm() < 1e-14)
final_velocity = gradient(superposition(inviscid[-1] / eigenvalues)) ^ normals  # Point-[F]
assert np.all((final_velocity & patches).norm() < 1e-13)
# On every triangle, the Jacobian wedge pairing equals the area-weighted directional derivative.
n_checked_modes = 6
face_gradients = gradient(modes[:n_checked_modes])                              # [n_checked_modes] Plane+[F]
wedge_brackets = (face_gradients[:, None] ^ face_gradients[None, :]) & patches  # [n_checked_modes, n_checked_modes] Scalar+[F]
directional = (face_gradients[:, None] & (face_gradients[None, :] ^ normals)) * areas   # [n_checked_modes, n_checked_modes] Scalar+[F]
assert np.all((wedge_brackets - directional).abs() < 1e-14)
# The Lie–Poisson structure constants are totally antisymmetric in all three mode indices to round-off.
constants = structure.kernel[..., 0, 0, 0]                                      # [n_modes, n_modes, n_modes]
np.testing.assert_allclose(constants + constants.swapaxes(1, 2), 0.0, atol=1e-13)
np.testing.assert_allclose(constants + constants.swapaxes(0, 1), 0.0, atol=1e-13)
# Instantaneous inviscid advection conserves kinetic energy and enstrophy, and RK4 keeps both flat.
initial_rate = vorticity_rate(start, structure, eigenvalues, 0.0)               # Scalar+[n_modes]
assert (((start / eigenvalues) * initial_rate).sites.sum()).abs() < 1e-11
assert ((start * initial_rate).sites.sum()).abs() < 1e-11
assert np.all((inviscid_energy - inviscid_energy[0]).abs() < 5e-5 * inviscid_energy[0])
assert np.all((inviscid_enstrophy - inviscid_enstrophy[0]).abs() < 5e-5 * inviscid_enstrophy[0])
# Viscosity dissipates both invariants, damping high-wavenumber enstrophy faster than kinetic energy.
assert np.all(viscous_energy[1:] < viscous_energy[:-1])
assert np.all(viscous_enstrophy[1:] < viscous_enstrophy[:-1])
assert viscous_enstrophy[-1] / viscous_enstrophy[0] < viscous_energy[-1] / viscous_energy[0]